# 01 — Dataset acquisition and understanding

## Prediction task
Using information available at discharge, predict whether a hospital encounter involving a patient with diabetes is followed by a recorded readmission within 30 days.

- Unit: one hospital encounter.
- Target: `<30` = 1; `>30` and `NO` = 0. This follows the source labels; `NO` does not establish complete follow-up across all hospitals.
- Prediction time: discharge.
- Retain `patient_nbr` for grouped splitting; exclude both identifiers and the original `readmitted` column from model inputs.
- Keep every encounter for this initial audit. Decide the eligible discharge cohort before modeling, including how to handle deaths and hospice discharges.
- Later, reserve patient-disjoint train/validation/test partitions before fitting imputers, encoders, feature selection, or resampling. Training-only exploration should guide modeling decisions.

## Source
[UCI Diabetes 130-US Hospitals for Years 1999–2008](https://archive.ics.uci.edu/dataset/296/diabetes+130+us+hospitals+for+years+1999+2008).
Clore, J., Cios, K., DeShazo, J., & Strack, B. (2014). DOI: https://doi.org/10.24432/C5230J. License: CC BY 4.0.
Historical US diabetes encounter data; this project is educational and is not clinically validated.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw' / 'diabetic_data.csv').exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data' / 'raw' / 'diabetic_data.csv',
                 keep_default_na=False, na_values=['?'], low_memory=False)
assert df['readmitted'].isin(['<30', '>30', 'NO']).all()
assert df['patient_nbr'].notna().all()
print('Raw shape:', df.shape)
print('Unique patients:', df['patient_nbr'].nunique())
print('Duplicate encounter IDs:', df['encounter_id'].duplicated().sum())
print(df.head().to_string(index=False))


Raw shape: (101766, 50)
Unique patients: 71518
Duplicate encounter IDs: 0
 encounter_id  patient_nbr            race gender     age weight  admission_type_id  discharge_disposition_id  admission_source_id  time_in_hospital payer_code        medical_specialty  num_lab_procedures  num_procedures  num_medications  number_outpatient  number_emergency  number_inpatient diag_1 diag_2 diag_3  number_diagnoses max_glu_serum A1Cresult metformin repaglinide nateglinide chlorpropamide glimepiride acetohexamide glipizide glyburide tolbutamide pioglitazone rosiglitazone acarbose miglitol troglitazone tolazamide examide citoglipton insulin glyburide-metformin glipizide-metformin glimepiride-pioglitazone metformin-rosiglitazone metformin-pioglitazone change diabetesMed readmitted
      2278392      8222157       Caucasian Female  [0-10)    NaN                  6                        25                    1                 1        NaN Pediatrics-Endocrinology                  41               0    

In [2]:
print('Column types and completeness:')
df.info()
schema = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'unique_values': df.nunique(dropna=False),
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean().mul(100),
})
print(schema.to_string())
print('\nNumeric summaries (identifier summaries are not predictive features):')
print(df.describe().to_string())


Column types and completeness:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            10

## Missing-value interpretation
Only `?` is converted to missing on load. In `A1Cresult` and `max_glu_serum`, the literal category `None` means no test result recorded and is preserved. Admission/discharge mapping codes also include unknown or unavailable categories; inspect these before preprocessing. No imputation or column removal is performed here.


In [3]:
print('Missing values (%):')
print(schema['missing_percent'].sort_values(ascending=False).to_string())
for column in ['A1Cresult', 'max_glu_serum', 'gender']:
    print('\n' + column)
    print(df[column].value_counts(dropna=False).to_string())
print('\nAdmission/discharge mapping file:')
print((ROOT / 'data' / 'raw' / 'IDS_mapping.csv').read_text())


Missing values (%):
weight                      96.858479
medical_specialty           49.082208
payer_code                  39.557416
race                         2.233555
diag_3                       1.398306
diag_2                       0.351787
diag_1                       0.020636
admission_type_id            0.000000
patient_nbr                  0.000000
encounter_id                 0.000000
time_in_hospital             0.000000
admission_source_id          0.000000
num_lab_procedures           0.000000
num_procedures               0.000000
num_medications              0.000000
discharge_disposition_id     0.000000
gender                       0.000000
age                          0.000000
number_inpatient             0.000000
number_emergency             0.000000
number_outpatient            0.000000
number_diagnoses             0.000000
max_glu_serum                0.000000
A1Cresult                    0.000000
metformin                    0.000000
repaglinide                  0

In [4]:
target = df['readmitted'].eq('<30').astype('int8').rename('target')
label_counts = df['readmitted'].value_counts()
print('Original readmission labels:')
print(label_counts.to_string())
print('\nBinary target:')
print(pd.DataFrame({'count': target.value_counts().sort_index(),
                    'percent': target.value_counts(normalize=True).sort_index() * 100}).to_string())
patient_counts = df.groupby('patient_nbr').size()
print('\nPatients with multiple encounters:', int(patient_counts.gt(1).sum()))
print('Additional encounters beyond one per patient:', len(df) - len(patient_counts))
print('Maximum encounters for one patient:', int(patient_counts.max()))
print('\nDischarge disposition counts (review eligibility before modeling):')
print(df['discharge_disposition_id'].value_counts().sort_index().to_string())


Original readmission labels:
readmitted
NO     54864
>30    35545
<30    11357

Binary target:
        count    percent
target                  
0       90409  88.840084
1       11357  11.159916

Patients with multiple encounters: 16773
Additional encounters beyond one per patient: 30248
Maximum encounters for one patient: 40

Discharge disposition counts (review eligibility before modeling):
discharge_disposition_id
1     60234
2      2128
3     13954
4       815
5      1184
6     12902
7       623
8       108
9        21
10        6
11     1642
12        3
13      399
14      372
15       63
16       11
17       14
18     3691
19        8
20        2
22     1993
23      412
24       48
25      989
27        5
28      139


In [5]:
reports = ROOT / 'reports'
reports.mkdir(exist_ok=True)
schema.to_csv(reports / 'column_inventory.csv', index_label='column')
summary = {
    'encounters': len(df), 'raw_columns': len(df.columns),
    'unique_patients': int(df['patient_nbr'].nunique()),
    'patients_with_multiple_encounters': int(patient_counts.gt(1).sum()),
    'duplicate_encounter_ids': int(df['encounter_id'].duplicated().sum()),
    'readmission_labels': {str(k): int(v) for k, v in label_counts.items()},
    'positive_target_count': int(target.sum()),
    'positive_target_percent': float(target.mean() * 100),
}
(reports / 'data_summary.json').write_text(__import__('json').dumps(summary, indent=2))
print(__import__('json').dumps(summary, indent=2))


{
  "encounters": 101766,
  "raw_columns": 50,
  "unique_patients": 71518,
  "patients_with_multiple_encounters": 16773,
  "duplicate_encounter_ids": 0,
  "readmission_labels": {
    "NO": 54864,
    ">30": 35545,
    "<30": 11357
  },
  "positive_target_count": 11357,
  "positive_target_percent": 11.159915885462727
}


## First milestone completed
The raw data is loaded, columns and missingness are inventoried, the target is defined, and repeated patients are measured. Raw files remain unchanged. No model has been trained and no preprocessing has been fitted.

Next: review discharge eligibility and coded missing categories, then create reproducible patient-disjoint partitions. The raw CSV contains 50 columns (including identifiers and target), while UCI lists 47 features.
